# Echogram Visualizations for Deep Learning

Exploring echogram visualization and downsampling strategies for U-Net input preparation.

In [ ]:
import sys
sys.path.insert(0, '../..')

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from raw_sensor_integration import ping_parser

## Survey Available Raw Files

Check multiple files to find one with good data quality.

In [ ]:
RAW_DIR = Path.home() / 'Downloads' / 'drive-download-20251116T003920Z-1-001'
raw_files = sorted(RAW_DIR.glob('*.raw'))

print(f"Found {len(raw_files)} raw files:\n")
for i, f in enumerate(raw_files):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"[{i}] {f.name} ({size_mb:.1f} MB)")

In [ ]:
def quick_preview(raw_file, max_pings=500):
    """Quick preview of a raw file to assess data quality."""
    pings = ping_parser.extract_pings(raw_file, max_pings=max_pings)
    
    # Get channels
    channels = {}
    for p in pings:
        if p.channel_id not in channels:
            channels[p.channel_id] = []
        channels[p.channel_id].append(p)
    
    print(f"\n{raw_file.name}")
    print(f"  Total pings: {len(pings)}")
    print(f"  Channels: {len(channels)}")
    
    for ch_id, ch_pings in channels.items():
        power_db = ping_parser.pings_to_power_image(pings, ch_id)
        valid_mask = power_db > -150  # Filter out placeholder values
        if valid_mask.any():
            p5, p50, p95 = np.percentile(power_db[valid_mask], [5, 50, 95])
            print(f"  {ch_id[:30]}...")
            print(f"    Shape: {power_db.shape}, dB range: {p5:.0f} / {p50:.0f} / {p95:.0f} (5/50/95 %ile)")
    
    return pings, channels

In [ ]:
# Preview a few files to find good data
for idx in [0, 2, 5, 8]:
    if idx < len(raw_files):
        try:
            quick_preview(raw_files[idx], max_pings=200)
        except Exception as e:
            print(f"[{idx}] {raw_files[idx].name}: Error - {e}")

## Load Selected File

Choose a file with good data quality based on the preview above.

In [ ]:
# Select file index (adjust based on preview results)
FILE_IDX = 0
MAX_PINGS = 1000*5

sample_file = raw_files[FILE_IDX]
print(f"Loading: {sample_file.name}")

pings = ping_parser.extract_pings(sample_file, max_pings=MAX_PINGS)
print(f"Extracted {len(pings)} pings")

# Organize by channel
channels = {}
for p in pings:
    if p.channel_id not in channels:
        channels[p.channel_id] = []
    channels[p.channel_id].append(p)

print("\nAvailable channels:")
for i, ch in enumerate(channels.keys()):
    print(f"  [{i}] {ch} ({len(channels[ch])} pings)")

In [ ]:
import re

# Determine frequency for each channel by scanning the XML config in the file
raw_bytes = open(sample_file, 'rb').read()
freq_pattern = re.compile(rb'Frequency="([^"]+)"')

def get_channel_freq_khz(channel_id: str, raw_bytes: bytes):
    """Return transmit frequency in kHz for a given channel, parsed from XML."""
    ch_bytes = channel_id.encode('ascii')
    pos = raw_bytes.find(ch_bytes)
    if pos == -1:
        return None
    window = raw_bytes[max(0, pos - 500): pos + 2500]
    matches = freq_pattern.findall(window)
    freqs = [float(m) for m in matches if float(m) > 1000]
    return min(freqs) / 1000 if freqs else None

# Build channel → frequency label mapping
ch_freq = {}
for ch_id in channels:
    f_khz = get_channel_freq_khz(ch_id, raw_bytes)
    if f_khz:
        ch_freq[ch_id] = f"{f_khz:.0f} kHz"
    else:
        ch_freq[ch_id] = ch_id.split(' ')[1] if ' ' in ch_id else ch_id

print("Channel → Frequency mapping:")
for ch_id, label in ch_freq.items():
    print(f"  {ch_id}  →  {label}")

In [ ]:
# Build power images for all channels
ch_power = {}
for ch_id in channels:
    img = ping_parser.pings_to_power_image(pings, ch_id)
    img = np.clip(img, -100, 0)
    ch_power[ch_id] = img

# Plot all channels side by side
n_ch = len(channels)
fig, axes = plt.subplots(1, n_ch, figsize=(7 * n_ch, 7), sharey=False)
if n_ch == 1:
    axes = [axes]

for ax, ch_id in zip(axes, channels):
    img = ch_power[ch_id]
    label = ch_freq.get(ch_id, ch_id)
    vmin_ch, vmax_ch = np.percentile(img, [2, 98])

    im = ax.imshow(img, aspect='auto', cmap='viridis', vmin=vmin_ch, vmax=vmax_ch, origin='upper')
    ax.set_title(f"{label}\n{img.shape[0]} depth samples × {img.shape[1]} pings", fontsize=11)
    ax.set_xlabel('Ping Number')
    ax.set_ylabel('Depth Sample')
    cbar = plt.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label('Power (dB)')

plt.suptitle(f"All Frequencies — {sample_file.name}", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Multi-Frequency Overview

The EK80 records multiple frequencies simultaneously. Below we compare single-ping
depth profiles across all available frequencies.

In [ ]:
# Single ping profiles for all frequencies — compare how each frequency sees the water column
PING_IDX = 250  # Pick a middle ping

fig, ax = plt.subplots(figsize=(8, 7))

colors = ['steelblue', 'tomato', 'forestgreen', 'darkorange']
for (ch_id, img), color in zip(ch_power.items(), colors):
    if PING_IDX >= img.shape[1]:
        continue
    profile = img[:, PING_IDX]
    depth_idx = np.arange(len(profile))
    label = ch_freq.get(ch_id, ch_id)
    ax.plot(profile, depth_idx, color=color, lw=1.5, label=label)

ax.invert_yaxis()
ax.set_xlabel('Power (dB)')
ax.set_ylabel('Depth Sample')
ax.set_title(f'Ping {PING_IDX} — All Frequencies')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Select channel
CHANNEL_IDX = 0
selected_channel = list(channels.keys())[CHANNEL_IDX]

# Build power image
power_db = ping_parser.pings_to_power_image(pings, selected_channel)

# Clean up extreme values (replace -200 placeholders with reasonable minimum)
power_db = np.clip(power_db, -100, 0)

print(f"Channel: {selected_channel}")
print(f"Image shape: {power_db.shape} (depth samples x pings)")
print(f"Power range: {power_db.min():.1f} to {power_db.max():.1f} dB")

## Basic Echogram Visualization

Standard visualization with sensible defaults for acoustic data.

In [ ]:
from raw_sensor_integration.visualization import plot_echogram

In [ ]:
# Determine good visualization range from data
vmin, vmax = np.percentile(power_db, [2, 98])
print(f"Auto range: {vmin:.1f} to {vmax:.1f} dB")

plot_echogram(power_db, "Original Echogram", vmin=vmin, vmax=vmax)
plt.tight_layout()
plt.show()

## Colormap Comparison

Different colormaps for acoustic interpretation.

In [ ]:
# Colormaps commonly used for echograms
cmaps = ['viridis', 'cividis', 'plasma', 'inferno', 'gray_r', 'bone_r']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, cmap in enumerate(cmaps):
    im = axes[i].imshow(power_db, aspect='auto', cmap=cmap, vmin=vmin, vmax=vmax)
    axes[i].set_xlabel('Ping')
    axes[i].set_ylabel('Depth')
    axes[i].set_title(cmap)
    plt.colorbar(im, ax=axes[i], shrink=0.7)

plt.tight_layout()
plt.show()

## Depth Downsampling for U-Net

Adjacent depth samples are highly correlated, so we can reduce the depth dimension
without losing much information. Ping-to-ping variation matters more for the model,
so we keep that axis at full resolution.

Three pooling strategies: max (preserves strong returns like bottom/fish),
mean (smooths noise), and simple decimation (every Nth sample).

In [ ]:
from raw_sensor_integration.preprocessing import downsample_depth

In [ ]:
# Compare downsampling factors (depth only)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

configs = [
    (1, 'Original'),
    (2, 'Half height (2x)'),
    (4, 'Quarter height (4x)'),
    (8, 'Eighth height (8x)')
]

for ax, (factor, label) in zip(axes.flatten(), configs):
    if factor == 1:
        img = power_db
    else:
        img = downsample_depth(power_db, factor, method='max')
    
    im = ax.imshow(img, aspect='auto', cmap='viridis', vmin=vmin, vmax=vmax)
    ax.set_xlabel('Ping')
    ax.set_ylabel('Depth')
    ax.set_title(f"{label}: {img.shape[0]} x {img.shape[1]}")
    plt.colorbar(im, ax=ax, shrink=0.7)

plt.tight_layout()
plt.show()

## Pooling Method Comparison

Compare max vs mean pooling for depth downsampling.

In [ ]:
FACTOR = 4  # Quarter height

down_max = downsample_depth(power_db, FACTOR, method='max')
down_mean = downsample_depth(power_db, FACTOR, method='mean')
down_decimate = downsample_depth(power_db, FACTOR, method='decimate')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Original
im0 = axes[0, 0].imshow(power_db, aspect='auto', cmap='viridis', vmin=vmin, vmax=vmax)
axes[0, 0].set_title(f"Original: {power_db.shape[0]} x {power_db.shape[1]}")
axes[0, 0].set_xlabel('Ping')
axes[0, 0].set_ylabel('Depth')
plt.colorbar(im0, ax=axes[0, 0], shrink=0.7)

# Max pooling
im1 = axes[0, 1].imshow(down_max, aspect='auto', cmap='viridis', vmin=vmin, vmax=vmax)
axes[0, 1].set_title(f"Max Pool {FACTOR}x: {down_max.shape[0]} x {down_max.shape[1]}")
axes[0, 1].set_xlabel('Ping')
axes[0, 1].set_ylabel('Depth')
plt.colorbar(im1, ax=axes[0, 1], shrink=0.7)

# Mean pooling
im2 = axes[1, 0].imshow(down_mean, aspect='auto', cmap='viridis', vmin=vmin, vmax=vmax)
axes[1, 0].set_title(f"Mean Pool {FACTOR}x: {down_mean.shape[0]} x {down_mean.shape[1]}")
axes[1, 0].set_xlabel('Ping')
axes[1, 0].set_ylabel('Depth')
plt.colorbar(im2, ax=axes[1, 0], shrink=0.7)

# Decimate
im3 = axes[1, 1].imshow(down_decimate, aspect='auto', cmap='viridis', vmin=vmin, vmax=vmax)
axes[1, 1].set_title(f"Decimate {FACTOR}x: {down_decimate.shape[0]} x {down_decimate.shape[1]}")
axes[1, 1].set_xlabel('Ping')
axes[1, 1].set_ylabel('Depth')
plt.colorbar(im3, ax=axes[1, 1], shrink=0.7)

plt.tight_layout()
plt.show()

## Detail Preservation Check

Zoom into a region to verify features are preserved after downsampling.

In [ ]:
# Select a region of interest (adjust based on your data)
# These are in original coordinates
DEPTH_START = power_db.shape[0] // 4
DEPTH_END = power_db.shape[0] // 2
PING_START = 100
PING_END = 300

# Extract regions
roi_original = power_db[DEPTH_START:DEPTH_END, PING_START:PING_END]

# Corresponding region in downsampled (factor 4)
FACTOR = 4
down_full = downsample_depth(power_db, FACTOR, method='max')
roi_down = down_full[DEPTH_START//FACTOR:DEPTH_END//FACTOR, PING_START:PING_END]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im0 = axes[0].imshow(roi_original, aspect='auto', cmap='viridis', vmin=vmin, vmax=vmax)
axes[0].set_title(f"Original ROI: {roi_original.shape[0]} x {roi_original.shape[1]}")
axes[0].set_xlabel('Ping')
axes[0].set_ylabel('Depth')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(roi_down, aspect='auto', cmap='viridis', vmin=vmin, vmax=vmax)
axes[1].set_title(f"Downsampled {FACTOR}x ROI: {roi_down.shape[0]} x {roi_down.shape[1]}")
axes[1].set_xlabel('Ping')
axes[1].set_ylabel('Depth')
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

## Single Ping Profile Comparison

Compare how a single ping looks before and after downsampling.

In [ ]:
PING_IDX = power_db.shape[1] // 2  # Middle ping
FACTOR = 4

original_profile = power_db[:, PING_IDX]
down_max_profile = downsample_depth(power_db, FACTOR, method='max')[:, PING_IDX]
down_mean_profile = downsample_depth(power_db, FACTOR, method='mean')[:, PING_IDX]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full profile
depth_orig = np.arange(len(original_profile))
depth_down = np.arange(len(down_max_profile)) * FACTOR

axes[0].plot(original_profile, depth_orig, 'b-', alpha=0.7, label='Original', linewidth=0.5)
axes[0].plot(down_max_profile, depth_down, 'r-', label=f'Max pool {FACTOR}x', linewidth=1.5)
axes[0].plot(down_mean_profile, depth_down, 'g--', label=f'Mean pool {FACTOR}x', linewidth=1.5)
axes[0].set_xlabel('Power (dB)')
axes[0].set_ylabel('Depth Sample')
axes[0].set_title(f'Ping {PING_IDX} - Full Profile')
axes[0].legend()
axes[0].invert_yaxis()
axes[0].set_xlim(vmin, vmax)
axes[0].grid(True, alpha=0.3)

# Zoomed region
zoom_start = len(original_profile) // 3
zoom_end = len(original_profile) // 2

axes[1].plot(original_profile[zoom_start:zoom_end], depth_orig[zoom_start:zoom_end], 
             'b-', alpha=0.7, label='Original', linewidth=0.8)
axes[1].plot(down_max_profile[zoom_start//FACTOR:zoom_end//FACTOR], 
             depth_down[zoom_start//FACTOR:zoom_end//FACTOR], 
             'r-', label=f'Max pool {FACTOR}x', linewidth=2)
axes[1].set_xlabel('Power (dB)')
axes[1].set_ylabel('Depth Sample')
axes[1].set_title('Zoomed Region')
axes[1].legend()
axes[1].invert_yaxis()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Recommended U-Net Input Sizes

U-Net works best with dimensions that are powers of 2 (or at least divisible by 16/32).

In [ ]:
def suggest_unet_size(h, w, target_h_range=(256, 1024)):
    """Suggest downsampling factors for U-Net friendly sizes."""
    print(f"Original size: {h} x {w}")
    print("\nSuggested downsampling (depth only):")
    
    for factor in [1, 2, 4, 8]:
        new_h = h // factor
        # Round to nearest multiple of 32 for U-Net
        unet_h = (new_h // 32) * 32
        
        status = "good" if target_h_range[0] <= unet_h <= target_h_range[1] else "-"
        print(f"  [{status}] Factor {factor}x: {new_h} -> {unet_h} (rounded to 32)")

suggest_unet_size(power_db.shape[0], power_db.shape[1])

In [ ]:
# Create U-Net ready input
DEPTH_FACTOR = 4  # Adjust based on suggestions above
TARGET_HEIGHT = 512  # Or 256, 384, etc.

# Downsample
downsampled = downsample_depth(power_db, DEPTH_FACTOR, method='max')

# Crop or pad to target height
current_h = downsampled.shape[0]
if current_h > TARGET_HEIGHT:
    # Center crop
    start = (current_h - TARGET_HEIGHT) // 2
    unet_input = downsampled[start:start + TARGET_HEIGHT, :]
else:
    # Pad with minimum value
    pad_total = TARGET_HEIGHT - current_h
    pad_top = pad_total // 2
    pad_bottom = pad_total - pad_top
    unet_input = np.pad(downsampled, ((pad_top, pad_bottom), (0, 0)), 
                        mode='constant', constant_values=power_db.min())

print(f"Original: {power_db.shape}")
print(f"After {DEPTH_FACTOR}x depth downsample: {downsampled.shape}")
print(f"U-Net input: {unet_input.shape}")

In [ ]:
# Final comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im0 = axes[0].imshow(power_db, aspect='auto', cmap='viridis', vmin=vmin, vmax=vmax)
axes[0].set_title(f"Original: {power_db.shape[0]} x {power_db.shape[1]}")
axes[0].set_xlabel('Ping')
axes[0].set_ylabel('Depth Sample')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(unet_input, aspect='auto', cmap='viridis', vmin=vmin, vmax=vmax)
axes[1].set_title(f"U-Net Input: {unet_input.shape[0]} x {unet_input.shape[1]}")
axes[1].set_xlabel('Ping')
axes[1].set_ylabel('Depth Sample')
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

## Normalization for Neural Network

Prepare data for neural network input (normalize to 0-1 or -1 to 1 range).

In [ ]:
from raw_sensor_integration.preprocessing import normalize_echogram


# Compare normalization methods
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, method in zip(axes, ['minmax', 'symmetric', 'standard']):
    normalized = normalize_echogram(unet_input, method=method, clip_percentile=(2, 98))
    im = ax.imshow(normalized, aspect='auto', cmap='viridis')
    ax.set_title(f"{method}: [{normalized.min():.2f}, {normalized.max():.2f}]")
    ax.set_xlabel('Ping')
    ax.set_ylabel('Depth')
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

## Preprocessing Pipeline

The full pipeline for preparing echograms as U-Net input:

```python
# 1. Load and clean
power_db = load_echogram(raw_file)
power_db = np.clip(power_db, -100, 0)

# 2. Downsample depth (keep full ping resolution)
downsampled = downsample_depth(power_db, factor=4, method='max')

# 3. Crop/pad to U-Net friendly size
unet_input = crop_or_pad(downsampled, target_height=512)

# 4. Normalize
normalized = normalize_for_nn(unet_input, method='minmax')

# 5. Add channel dimension for PyTorch/TensorFlow
model_input = normalized[np.newaxis, :, :]  # (1, H, W)
```

In [ ]:
# Complete preprocessing example
def preprocess_for_unet(power_db, depth_factor=4, target_height=512, norm_method='minmax'):
    """Complete preprocessing pipeline for U-Net input."""
    # Clean
    cleaned = np.clip(power_db, -100, 0)
    
    # Downsample depth only
    downsampled = downsample_depth(cleaned, depth_factor, method='max')
    
    # Crop or pad height
    h = downsampled.shape[0]
    if h > target_height:
        start = (h - target_height) // 2
        cropped = downsampled[start:start + target_height, :]
    else:
        pad = target_height - h
        cropped = np.pad(downsampled, ((pad//2, pad - pad//2), (0, 0)), 
                        mode='constant', constant_values=-100)
    
    # Normalize
    normalized = normalize_echogram(cropped, method=norm_method, clip_percentile=(2, 98))
    
    return normalized


# Test the pipeline
processed = preprocess_for_unet(power_db, depth_factor=4, target_height=512)
print(f"Final shape: {processed.shape}")
print(f"Value range: [{processed.min():.3f}, {processed.max():.3f}]")

plt.figure(figsize=(12, 5))
plt.imshow(processed, aspect='auto', cmap='viridis')
plt.title(f'U-Net Ready: {processed.shape[0]} x {processed.shape[1]}')
plt.xlabel('Ping')
plt.ylabel('Depth')
plt.colorbar(label='Normalized')
plt.tight_layout()
plt.show()